# Kiểm thử Kịch bản 1 — Đánh giá độ chính xác NER (Exact Match & Fuzzy Match)

**Mục đích:** So sánh kỹ năng pipeline trích xuất tự động vs nhãn chuẩn gán tay,  
tính Confusion Matrix → Precision / Recall / F1-Score cho **JD** và **CV** theo 2 chế độ:
1. **Exact Match (Khắt khe):** So khớp chính xác 100% chuỗi ký tự.
2. **Fuzzy Match (Linh hoạt):** So khớp mờ độ tương đồng ký tự $\ge 80\%$ (sử dụng thư viện chuẩn `difflib`).

**Cải tiến nâng cao (True NER Accuracy):**
- Tự động đối chiếu các lỗi False Positive (FP - Pipeline có nhưng GT không có) với văn bản gốc của JD (trong `jds_raw_text.txt`) và CV (trong thư mục `cvs/`).
- **Giải pháp không phạt lỗi phân tích chi tiết (Keyword Grounding):** Thay vì check so khớp cứng cả cụm từ, hệ thống sẽ bóc tách các từ khóa chính của kỹ năng. Nếu từ khóa cốt lõi thực sự xuất hiện trong văn bản gốc (ví dụ: Pipeline trích `english reading comprehension` và văn bản gốc có chứa `english`), kỹ năng đó được coi là **hợp lệ** (miễn trừ lỗi FP, đưa vào danh sách **Omitted**).

---
## ⚙️ Cấu hình và load dữ liệu

In [1]:
import sys, os, json, shutil, re
from pathlib import Path
from difflib import SequenceMatcher

# ── Đường dẫn ──────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(".").resolve()           # KiemThu/KiemThu_LLM_Extract/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent    # JobVisualization_BE/
DATA_DIR     = NOTEBOOK_DIR

F_PIPELINE_JDS = DATA_DIR / "pipeline_jds_output.json"
F_PIPELINE_CVS = DATA_DIR / "pipeline_cvs_output.json"
F_GT_JDS       = DATA_DIR / "ground_truth_jds.json"
F_GT_CVS       = DATA_DIR / "ground_truth_cvs.json"
F_JDS_RAW      = DATA_DIR / "jds_raw_text.txt"
CV_DIR         = DATA_DIR / "cvs"

# Thêm project root vào sys.path để import matching_cv
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Kiểm tra file tồn tại ──────────────────────────────────────────────────
missing = [f for f in [F_PIPELINE_JDS, F_PIPELINE_CVS, F_GT_JDS, F_GT_CVS] if not f.exists()]
if missing:
    for f in missing:
        print(f"❌ Không tìm thấy: {f.name}")
    raise FileNotFoundError("Chạy extract_20_jds_cvs_skills.ipynb và gán nhãn ground truth trước.")

# ── Load dữ liệu ──────────────────────────────────────────────────────────
pipeline_jds = json.loads(F_PIPELINE_JDS.read_text(encoding="utf-8"))
pipeline_cvs = json.loads(F_PIPELINE_CVS.read_text(encoding="utf-8"))
gt_jds       = json.loads(F_GT_JDS.read_text(encoding="utf-8"))
gt_cvs       = json.loads(F_GT_CVS.read_text(encoding="utf-8"))

# In hộp dữ liệu nạp thành công dạng console
lines = [
    f"✓ pipeline_jds_output.json : {len(pipeline_jds)} bản ghi JD",
    f"✓ pipeline_cvs_output.json : {len(pipeline_cvs)} bản ghi CV",
    f"✓ ground_truth_jds.json    : {len(gt_jds)} bản ghi nhãn chuẩn (GT)",
    f"✓ ground_truth_cvs.json    : {len(gt_cvs)} bản ghi nhãn chuẩn (GT)"
]
max_w = max(len(l) for l in lines) + 4
border = "─" * (max_w + 2)
print(f"┌{border}┐")
print(f"│ {'NẠP DỮ LIỆU ĐỐI SOÁT THỬ NGHIỆM':^{max_w}} │")
print(f"├{border}┤")
for line in lines:
    print(f"│  {line:<{max_w}}  │")
print(f"└{border}┘")


┌─────────────────────────────────────────────────────────────┐
│               NẠP DỮ LIỆU ĐỐI SOÁT THỬ NGHIỆM               │
├─────────────────────────────────────────────────────────────┤
│  ✓ pipeline_jds_output.json : 20 bản ghi JD                   │
│  ✓ pipeline_cvs_output.json : 20 bản ghi CV                   │
│  ✓ ground_truth_jds.json    : 20 bản ghi nhãn chuẩn (GT)      │
│  ✓ ground_truth_cvs.json    : 20 bản ghi nhãn chuẩn (GT)      │
└─────────────────────────────────────────────────────────────┘


---
## 📖 Đọc văn bản thô (JD & CV) phục vụ kiểm chứng chéo

In [2]:
# 1. Đọc và phân tích file jds_raw_text.txt
def parse_raw_jds_text(filepath):
    if not filepath.exists():
        return {}
    content = filepath.read_text(encoding="utf-8")
    blocks = content.split("=" * 80)
    url_to_text = {}
    for block in blocks:
        if not block.strip():
            continue
        url_match = re.search(r"URL\s*:\s*(https?://[^\s\r\n]+)", block)
        if url_match:
            url = url_match.group(1).strip()
            parts = re.split(r"─{10,}", block)
            text_content = parts[1].strip() if len(parts) > 1 else block
            url_to_text[url] = text_content
    return url_to_text

jds_raw_texts = parse_raw_jds_text(F_JDS_RAW)
print(f"✓ Đã nạp văn bản thô của {len(jds_raw_texts)} JD từ jds_raw_text.txt")

# 2. Đọc văn bản thô từ các file CV thực tế
cvs_raw_texts = {}
try:
    from matching_cv.utils import extract_cv_text
    ALLOWED_EXT = {".pdf", ".jpg", ".jpeg", ".png"}
    cv_files = sorted([
        f for f in CV_DIR.rglob("*")
        if f.is_file() and f.suffix.lower() in ALLOWED_EXT
    ])
    for f in cv_files:
        try:
            text = extract_cv_text(str(f))
            cvs_raw_texts[f.name] = text
        except Exception:
            pass
    print(f"✓ Đã trích xuất văn bản thô của {len(cvs_raw_texts)} file CV trong cvs/")
except Exception as e:
    print(f"⚠️ Không thể import module matching_cv hoặc đọc CV: {e}")

✓ Đã nạp văn bản thô của 20 JD từ jds_raw_text.txt
✓ Đã trích xuất văn bản thô của 11 file CV trong cvs/


---
## 🔧 Hàm tính Confusion Matrix cải tiến

In [3]:
def normalize_skill(s: str) -> str:
    """Chuẩn hóa skill về lowercase để so sánh không phân biệt hoa/thường."""
    return str(s).strip().lower()


def calc_metrics(tp: int, fp: int, fn: int) -> tuple:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return round(precision, 4), round(recall, 4), round(f1, 4)


def is_fuzzy_match(s1: str, s2: str, threshold: float = 0.8) -> bool:
    """Kiểm tra độ tương đồng chuỗi ký tự."""
    ratio = SequenceMatcher(None, s1, s2).ratio()
    if s1 in s2 or s2 in s1:
        len_ratio = min(len(s1), len(s2)) / max(len(s1), len(s2))
        if len_ratio >= 0.5:
            return True
    return ratio >= threshold


def is_skill_in_raw_text(skill: str, raw_text: str) -> bool:
    """Kiểm tra xem kỹ năng có thực sự xuất hiện trong văn bản gốc không (Theo từ khóa cốt lõi)."""
    if not raw_text:
        return False
    skill_cleaned = skill.strip().lower()
    text_cleaned = raw_text.strip().lower()
    
    # 1. So khớp trực tiếp cụm từ
    if skill_cleaned in text_cleaned:
        return True
        
    # 2. So khớp theo từ khóa chính (loại bỏ các stop-words chung chung)
    stop_words = {
        'and', 'of', 'in', 'or', 'skill', 'skills', 'methodology', 'comprehension', 
        'ability', 'proficiency', 'knowledge', 'development', 'management', 
        'engineering', 'system', 'systems', 'platform', 'platforms', 'tool', 
        'tools', 'domain', 'practices', 'solutions', 'certification', 'certifications'
    }
    words = [w for w in re.split(r"\W+", skill_cleaned) if w and w not in stop_words]
    
    if not words:
        return False
        
    # Kiểm tra xem có từ khóa cốt lõi nào xuất hiện độc lập trong văn bản gốc không
    for word in words:
        if len(word) >= 3:
            if bool(re.search(r"\b" + re.escape(word) + r"\b", text_cleaned)):
                return True
    return False


def compute_confusion_matrix(pipeline_items: list, gt_items: list, key_field: str, raw_texts_map: dict, fuzzy: bool = False) -> list:
    """
    So sánh pipeline output vs ground truth theo key_field ("url" hoặc "filename").
    Hỗ trợ đối chiếu văn bản thô để loại bỏ lỗi FP oan nếu kỹ năng thực sự có trong văn bản.
    """
    gt_map = {
        item[key_field]: [normalize_skill(s) for s in item.get("skills", [])]
        for item in gt_items
        if item.get(key_field)
    }

    results = []
    for item in pipeline_items:
        key   = item.get(key_field, "")
        label = key.split("/")[-1] if "/" in key else key
        label = label[:50] + "..." if len(label) > 50 else label

        p_skills  = [normalize_skill(s) for s in item.get("skills", [])]
        gt_skills_list = gt_map.get(key, None)
        raw_text = raw_texts_map.get(key, "")

        if gt_skills_list is None:
            results.append({"key": label, "status": "MISSING_GT",
                            "tp": 0, "fp": 0, "fn": 0,
                            "precision": 0, "recall": 0, "f1": 0,
                            "tp_skills": [], "fp_skills": [], "fn_skills": [], "omitted_skills": []})
            continue

        # Tính toán các tập kỹ năng thô
        if not fuzzy:
            p_set = set(p_skills)
            gt_set = set(gt_skills_list)
            tp_set = p_set & gt_set
            raw_fp_set = p_set - gt_set
            fn_set = gt_set - p_set
        else:
            tp_set = set()
            raw_fp_set = set()
            remaining_gt = list(gt_skills_list)
            for p_skill in p_skills:
                matched = False
                for gt_skill in list(remaining_gt):
                    if is_fuzzy_match(p_skill, gt_skill, threshold=0.8):
                        tp_set.add(p_skill)
                        remaining_gt.remove(gt_skill)
                        matched = True
                        break
                if not matched:
                    raw_fp_set.add(p_skill)
            fn_set = set(remaining_gt)

        # 🔎 ĐỐI CHIẾU VĂN BẢN GỐC (Lọc bỏ FP oan)
        fp_set = set()
        omitted_set = set()
        for fp_skill in raw_fp_set:
            if is_skill_in_raw_text(fp_skill, raw_text):
                # Có trong văn bản gốc -> Kỹ năng đúng nhưng GT gán thiếu
                omitted_set.add(fp_skill)
            else:
                # Không có trong văn bản gốc -> Lỗi bịa thực sự
                fp_set.add(fp_skill)

        tp, fp, fn        = len(tp_set), len(fp_set), len(fn_set)
        precision, recall, f1 = calc_metrics(tp, fp, fn)

        results.append({
            "key":            label,
            "status":         "OK",
            "tp":             tp,  "fp":       fp,  "fn":       fn,
            "precision":      precision, "recall": recall, "f1": f1,
            "tp_skills":      sorted(tp_set),
            "fp_skills":      sorted(fp_set),
            "fn_skills":      sorted(fn_set),
            "omitted_skills": sorted(omitted_set),
        })

    return results


def print_summary_table(results, title_label):
    W = 106
    title_padded = f" BẢNG ĐÁNH GIÁ: {title_label} "
    print("┌" + "─"*(W-2) + "┐")
    print(f"│{title_padded:^{W-2}}│")
    print("├" + "─"*6 + "┬" + "─"*48 + "┬" + "─"*5 + "┬" + "─"*5 + "┬" + "─"*5 + "┬" + "─"*9 + "┬" + "─"*8 + "┬" + "─"*8 + "┬" + "─"*8 + "┤")
    print(f"│ {'STT':<4} │ {'JD/CV (Tên rút gọn)':<46} │ {'TP':>3} │ {'FP':>3} │ {'FN':>3} │ {'Omitted':>7} │ {'Prec.':>6} │ {'Rec.':>6} │ {'F1':>6} │")
    print("├" + "─"*6 + "┼" + "─"*48 + "┼" + "─"*5 + "┼" + "─"*5 + "┼" + "─"*5 + "┼" + "─"*9 + "┼" + "─"*8 + "┼" + "─"*8 + "┼" + "─"*8 + "┤")

    total_tp = total_fp = total_fn = total_omitted = 0
    valid_count = 0

    for idx, r in enumerate(results, 1):
        if r["status"] == "MISSING_GT":
            print(f"│ {idx:<4} │ {r['key'][:46]:<46} │ {'— Chưa có ground truth':>48} │")
            continue
        total_tp += r["tp"]; total_fp += r["fp"]; total_fn += r["fn"]; total_omitted += len(r["omitted_skills"])
        valid_count += 1
        print(f"│ {idx:<4} │ {r['key'][:46]:<46} │ {r['tp']:>3} │ {r['fp']:>3} │ {r['fn']:>3} │ {len(r['omitted_skills']):>7} │ "
              f"{r['precision']*100:>5.1f}% │ {r['recall']*100:>5.1f}% │ {r['f1']*100:>5.1f}% │")

    print("├" + "─"*6 + "┼" + "─"*48 + "┼" + "─"*5 + "┼" + "─"*5 + "┼" + "─"*5 + "┼" + "─"*9 + "┼" + "─"*8 + "┼" + "─"*8 + "┼" + "─"*8 + "┤")
    ov_p, ov_r, ov_f1 = calc_metrics(total_tp, total_fp, total_fn)
    print(f"│ {'TỔNG':<4} │ {f'({valid_count} bản ghi đánh giá)':<46} │ "
          f"{total_tp:>3} │ {total_fp:>3} │ {total_fn:>3} │ {total_omitted:>7} │ "
          f"{ov_p*100:>5.1f}% │ {ov_r*100:>5.1f}% │ {ov_f1*100:>5.1f}% │")
    print("└" + "─"*(W-2) + "┘")
    print(f"  💡 Omitted: Số lượng kỹ năng có trong văn bản gốc nhưng nhãn Ground Truth gán thiếu (được miễn trừ lỗi FP).")
    return ov_p, ov_r, ov_f1

print("✓ Hàm tính Confusion Matrix và hiển thị bảng ASCII sẵn sàng")


✓ Hàm tính Confusion Matrix và hiển thị bảng ASCII sẵn sàng


---
## 📊 BẢNG 1 — Đánh giá trên tập JD (Tin tuyển dụng)

In [4]:
# Tính toán cả 2 chế độ cho JD
jd_results_exact = compute_confusion_matrix(pipeline_jds, gt_jds, key_field="url", raw_texts_map=jds_raw_texts, fuzzy=False)
jd_results_fuzzy = compute_confusion_matrix(pipeline_jds, gt_jds, key_field="url", raw_texts_map=jds_raw_texts, fuzzy=True)

print("1.1. CHẾ ĐỘ EXACT MATCH (KHẮT KHE - KHỚP 100% KÝ TỰ)")
jd_p_exact, jd_r_exact, jd_f1_exact = print_summary_table(jd_results_exact, "JD (TIN TUYỂN DỤNG) - EXACT MATCH")

print("\n1.2. CHẾ ĐỘ FUZZY MATCH (LINH HOẠT - CHẤP NHẬN TƯƠNG ĐỒNG KÝ TỰ >= 80%)")
jd_p_fuzzy, jd_r_fuzzy, jd_f1_fuzzy = print_summary_table(jd_results_fuzzy, "JD (TIN TUYỂN DỤNG) - FUZZY MATCH")


1.1. CHẾ ĐỘ EXACT MATCH (KHẮT KHE - KHỚP 100% KÝ TỰ)
┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                            BẢNG ĐÁNH GIÁ: JD (TIN TUYỂN DỤNG) - EXACT MATCH                            │
├──────┬────────────────────────────────────────────────┬─────┬─────┬─────┬─────────┬────────┬────────┬────────┤
│ STT  │ JD/CV (Tên rút gọn)                            │  TP │  FP │  FN │ Omitted │  Prec. │   Rec. │     F1 │
├──────┼────────────────────────────────────────────────┼─────┼─────┼─────┼─────────┼────────┼────────┼────────┤
│ 1    │ software-engineer-at-keyloop-4383379577?trk=pu │  28 │   0 │   7 │       7 │ 100.0% │  80.0% │  88.9% │
│ 2    │ software-engineer-at-fujifilm-business-innovat │  31 │   0 │   3 │       5 │ 100.0% │  91.2% │  95.4% │
│ 3    │ software-engineer-ii-at-global-fashion-group-4 │   8 │   1 │   2 │       2 │  88.9% │  80.0% │  84.2% │
│ 4    │ software-engineer-python-at-accenture-43702403

---
## 📊 BẢNG 2 — Đánh giá trên tập CV (Hồ sơ ứng viên)

In [5]:
# Tính toán cả 2 chế độ cho CV
cv_results_exact = compute_confusion_matrix(pipeline_cvs, gt_cvs, key_field="filename", raw_texts_map=cvs_raw_texts, fuzzy=False)
cv_results_fuzzy = compute_confusion_matrix(pipeline_cvs, gt_cvs, key_field="filename", raw_texts_map=cvs_raw_texts, fuzzy=True)

print("2.1. CHẾ ĐỘ EXACT MATCH (KHẮT KHE - KHỚP 100% KÝ TỰ)")
cv_p_exact, cv_r_exact, cv_f1_exact = print_summary_table(cv_results_exact, "CV (HỒ SƠ ỨNG VIÊN) - EXACT MATCH")

print("\n2.2. CHẾ ĐỘ FUZZY MATCH (LINH HOẠT - CHẤP NHẬN TƯƠNG ĐỒNG KÝ TỰ >= 80%)")
cv_p_fuzzy, cv_r_fuzzy, cv_f1_fuzzy = print_summary_table(cv_results_fuzzy, "CV (HỒ SƠ ỨNG VIÊN) - FUZZY MATCH")


2.1. CHẾ ĐỘ EXACT MATCH (KHẮT KHE - KHỚP 100% KÝ TỰ)
┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                            BẢNG ĐÁNH GIÁ: CV (HỒ SƠ ỨNG VIÊN) - EXACT MATCH                            │
├──────┬────────────────────────────────────────────────┬─────┬─────┬─────┬─────────┬────────┬────────┬────────┤
│ STT  │ JD/CV (Tên rút gọn)                            │  TP │  FP │  FN │ Omitted │  Prec. │   Rec. │     F1 │
├──────┼────────────────────────────────────────────────┼─────┼─────┼─────┼─────────┼────────┼────────┼────────┤
│ 1    │ 22127046_BaCong_CV.pdf                         │  13 │   3 │  12 │       1 │  81.2% │  52.0% │  63.4% │
│ 2    │ _CV_job__Quang_Huy_Tran.pdf                    │  29 │   2 │   2 │      29 │  93.5% │  93.5% │  93.5% │
│ 3    │ CareerNovaCV_HienLuong.pdf                     │  18 │   9 │   5 │      14 │  66.7% │  78.3% │  72.0% │
│ 4    │ CV_22127470 _LeHoangYen.pdf                   

---
## 🔍 So sánh và Kiểm tra ngưỡng kỳ vọng

In [6]:
def print_target_comparison(p, r, f1, label):
    W = 86
    inner_w = W - 2
    label_str = f" 📊 ĐÁNH GIÁ CHỈ SỐ: {label} "
    print("┌" + "─"*inner_w + "┐")
    print(f"│{label_str:^{inner_w}}│")
    print("├" + "─"*inner_w + "┤")
    print(f"│ Ngưỡng kỳ vọng: Precision >= 85% | Recall >= 80% | F1-Score >= 82.5%{' ':>20} │")
    print("├" + "─"*inner_w + "┤")
    
    def check(metric_name, val, target):
        status = "[  ✅ ĐẠT  ]" if val >= target else "[ ❌ CHƯA ĐẠT ]"
        content = f"  • {metric_name:<12} : {val*100:.2f}%  (mục tiêu >= {target*100:.1f}%)"
        pad_len = inner_w - len(status) - 4
        row = f"│ {content:<{pad_len}}  {status} │"
        print(row)
        
    check("Precision", p, 0.85)
    check("Recall", r, 0.80)
    check("F1-Score", f1, 0.825)
    print("└" + "─"*inner_w + "┘")
    print()

print("=== SO SÁNH TRÊN TẬP JD ===\n")
print_target_comparison(jd_p_exact, jd_r_exact, jd_f1_exact, "JD (EXACT MATCH - Đối chiếu văn bản gốc)")
print_target_comparison(jd_p_fuzzy, jd_r_fuzzy, jd_f1_fuzzy, "JD (FUZZY MATCH - Đối chiếu văn bản gốc)")

print("\n=== SO SÁNH TRÊN TẬP CV ===\n")
print_target_comparison(cv_p_exact, cv_r_exact, cv_f1_exact, "CV (EXACT MATCH - Đối chiếu văn bản gốc)")
print_target_comparison(cv_p_fuzzy, cv_r_fuzzy, cv_f1_fuzzy, "CV (FUZZY MATCH - Đối chiếu văn bản gốc)")


=== SO SÁNH TRÊN TẬP JD ===

┌────────────────────────────────────────────────────────────────────────────────────┐
│            📊 ĐÁNH GIÁ CHỈ SỐ: JD (EXACT MATCH - Đối chiếu văn bản gốc)             │
├────────────────────────────────────────────────────────────────────────────────────┤
│ Ngưỡng kỳ vọng: Precision >= 85% | Recall >= 80% | F1-Score >= 82.5%                     │
├────────────────────────────────────────────────────────────────────────────────────┤
│   • Precision    : 97.58%  (mục tiêu >= 85.0%)                         [  ✅ ĐẠT  ] │
│   • Recall       : 84.13%  (mục tiêu >= 80.0%)                         [  ✅ ĐẠT  ] │
│   • F1-Score     : 90.36%  (mục tiêu >= 82.5%)                         [  ✅ ĐẠT  ] │
└────────────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────────────┐
│            📊 ĐÁNH GIÁ CHỈ SỐ: JD (FUZZY MATCH - Đối chiếu văn bản gốc)             │
├──────

---
## 🔍 Chi tiết lỗi thực sự (Fuzzy Match - Đã trừ lỗi GT gán thiếu)

Xem chi tiết lỗi trích thừa (FP - bịa thực sự không có trong văn bản) và lỗi bỏ sót (FN).

In [7]:
import textwrap

def print_qualitative_errors_boxed(results, title_label):
    print(f"=== {title_label} ===")
    print()
    
    W = 86
    inner_w = W - 4
    
    for r in results:
        if r["status"] != "OK":
            continue
        if r["fp"] == 0 and r["fn"] == 0:
            continue
            
        print("┌" + "─"*(W-2) + "┐")
        doc_str = f"📄 {r['key']}"
        print(f"│ {doc_str:<{W-4}} │")
        print("├" + "─"*(W-2) + "┤")
        
        def print_wrapped_line(prefix, items, color_icon):
            if not items:
                return
            skills_str = ", ".join(items)
            label_w = 22
            wrapped_lines = textwrap.wrap(skills_str, width=inner_w - label_w)
            
            for idx, w_line in enumerate(wrapped_lines):
                if idx == 0:
                    line_text = f"{color_icon} {prefix:<17} : {w_line}"
                else:
                    line_text = f"{' ':<{label_w}}{w_line}"
                print(f"│ {line_text:<{W-4}} │")

        print_wrapped_line("FP (Trích thừa)", r["fp_skills"], "🔴")
        print_wrapped_line("FN (Bỏ sót thật)", r["fn_skills"], "🔵")
        
        if r["omitted_skills"]:
            skills_str = ", ".join(r["omitted_skills"]) + " (Đã miễn phạt)"
            label_w = 22
            wrapped_lines = textwrap.wrap(skills_str, width=inner_w - label_w)
            for idx, w_line in enumerate(wrapped_lines):
                if idx == 0:
                    line_text = f"💡 {'GT gán thiếu':<17} : {w_line}"
                else:
                    line_text = f"{' ':<{label_w}}{w_line}"
                print(f"│ {line_text:<{W-4}} │")
                
        print("└" + "─"*(W-2) + "┘")
        print()

print_qualitative_errors_boxed(jd_results_fuzzy, "CHI TIẾT LỖI SAI THỰC TẾ TRÊN JD (Fuzzy Match)")


=== CHI TIẾT LỖI SAI THỰC TẾ TRÊN JD (Fuzzy Match) ===

┌────────────────────────────────────────────────────────────────────────────────────┐
│ 📄 software-engineer-at-keyloop-4383379577?trk=public...                            │
├────────────────────────────────────────────────────────────────────────────────────┤
│ 🔵 FN (Bỏ sót thật)  : agile, dms, driven, language, oop                            │
│ 💡 GT gán thiếu      : agile development, dms domain, object-oriented programming   │
│                       principles, programming language proficiency, test-driven    │
│                       development (Đã miễn phạt)                                   │
└────────────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────────────┐
│ 📄 software-engineer-at-fujifilm-business-innovation-...                            │
├────────────────────────────────────────────────────────────────────────

In [8]:
print_qualitative_errors_boxed(cv_results_fuzzy, "CHI TIẾT LỖI SAI THỰC TẾ TRÊN CV (Fuzzy Match)")


=== CHI TIẾT LỖI SAI THỰC TẾ TRÊN CV (Fuzzy Match) ===

┌────────────────────────────────────────────────────────────────────────────────────┐
│ 📄 22127046_BaCong_CV.pdf                                                           │
├────────────────────────────────────────────────────────────────────────────────────┤
│ 🔴 FP (Trích thừa)   : ai models, computer vision                                   │
│ 🔵 FN (Bỏ sót thật)  : ai tạo sinh (generative ai), cấu trúc dữ liệu và giải thuật, │
│                       deep supervised auto-encoder hashing (saeh), ielts 7.0, lập  │
│                       trình back-end, mobile app development, mvssnet++, phân tích │
│                       và viết báo cáo khoa học, thuật toán xử lý ảnh y khoa, thị   │
│                       giác máy tính (computer vision), xử lý ảnh số và video số    │
│ 💡 GT gán thiếu      : hashing (Đã miễn phạt)                                       │
└─────────────────────────────────────────────────────────────────────────

---
## 💾 Lưu kết quả đánh giá

In [9]:
def summarize(results, label):
    valid = [r for r in results if r["status"] == "OK"]
    total_tp = sum(r["tp"] for r in valid)
    total_fp = sum(r["fp"] for r in valid)
    total_fn = sum(r["fn"] for r in valid)
    total_omitted = sum(len(r["omitted_skills"]) for r in valid)
    p, r, f1 = calc_metrics(total_tp, total_fp, total_fn)
    return {
        "dataset":         label,
        "count":           len(valid),
        "total_tp":        total_tp,
        "total_fp":        total_fp,
        "total_fn":        total_fn,
        "total_omitted":   total_omitted,
        "precision":       p,
        "recall":          r,
        "f1_score":        f1,
        "per_item":        valid,
    }

# Lưu kết quả của chế độ Fuzzy Match (vì nó phản ánh thực tế tốt nhất)
summary = {
    "jd": summarize(jd_results_fuzzy, "JD — Tin tuyển dụng (Fuzzy Match + Omitted Check)"),
    "cv": summarize(cv_results_fuzzy, "CV — Hồ sơ ứng viên (Fuzzy Match + Omitted Check)"),
}

f_result = DATA_DIR / "evaluation_result.json"
f_result.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✓ Đã lưu kết quả đánh giá (chế độ Fuzzy Match + Omitted Check): {f_result.name}")

✓ Đã lưu kết quả đánh giá (chế độ Fuzzy Match + Omitted Check): evaluation_result.json
